<a href="https://colab.research.google.com/github/lelongc/rac/blob/main/qwen_3_tts_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title ⚡️ SPEED PODCAST STUDIO (Hiển thị thời gian thực + Không lỗi 504)
import os
import time

# --- 1. CÀI ĐẶT THƯ VIỆN ---
print("⏳ Đang cài đặt thư viện (Đợi 1-2 phút)...")
os.system('pip install -U qwen-tts gradio huggingface_hub pydub')
os.system('apt-get install -y ffmpeg sox libsox-fmt-all')
print("✅ Cài đặt thư viện thành công!")

import gradio as gr
from qwen_tts import Qwen3TTSModel
import torch
import soundfile as sf
import tempfile
import gc
import re
from pydub import AudioSegment

# ================= QUẢN LÝ MODEL & BỘ NHỚ =================
torch.backends.cudnn.benchmark = True
current_model = None
current_model_type = None

def load_model_smart(task_type):
    global current_model, current_model_type

    if task_type == "DESIGN":
        model_name = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
    else:
        model_name = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"

    if current_model_type == task_type and current_model is not None:
        return current_model

    if current_model:
        del current_model
        gc.collect()
        torch.cuda.empty_cache()

    print(f"📥 Đang tải Model {task_type}... (Lần đầu mất 2-3 phút)")
    try:
        current_model = Qwen3TTSModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="cuda:0",
            attn_implementation="sdpa"
        )
        current_model_type = task_type
        print("✅ Model Ready!")
        return current_model
    except Exception as e:
        print(f"❌ Lỗi tải model: {e}")
        return None

# ================= TAB 1: TẠO MẪU GIỌNG =================
def create_voice_sample(prompt_text, sample_text):
    model = load_model_smart("DESIGN")
    if not model: return None

    with torch.inference_mode():
        w, sr = model.generate_voice_design(text=sample_text, instruct=prompt_text)

    temp_wav = tempfile.NamedTemporaryFile(delete=False, suffix=".wav").name
    sf.write(temp_wav, w[0], sr)
    return temp_wav

# ================= TAB 2: PODCAST SIÊU TỐC (CÓ YIELD CHỐNG 504) =================
def generate_podcast_fast(script_text, nam_ref_path, nu_ref_path):
    if not script_text or not nam_ref_path or not nu_ref_path:
        yield "⚠️ Lỗi: Vui lòng nhập đủ kịch bản và 2 file giọng mẫu!", None
        return

    yield "⏳ Đang khởi động AI & Tải file mẫu...", None
    model = load_model_smart("CLONE")
    if not model:
        yield "❌ Lỗi: Không thể tải Model.", None
        return

    final_audio = AudioSegment.silent(duration=500)
    gap = AudioSegment.silent(duration=400)

    lines = [l for l in script_text.strip().split('\n') if ":" in l]
    total_lines = len(lines)

    try:
        nam_prompt_feature = model.create_voice_clone_prompt(
            ref_audio=nam_ref_path, ref_text=None, x_vector_only_mode=True
        )
        nu_prompt_feature = model.create_voice_clone_prompt(
            ref_audio=nu_ref_path, ref_text=None, x_vector_only_mode=True
        )
    except Exception as e:
        yield f"❌ Lỗi đọc file mẫu: {e}", None
        return

    # Vòng lặp xịn: Vừa tạo audio, vừa báo cáo trạng thái ra web
    for index, line in enumerate(lines):
        name_part, text = line.split(":", 1)
        name_part = name_part.strip()
        text = text.strip()

        clean_name = re.sub(r'\(.*?\)', '', name_part).strip().lower()
        is_nam = clean_name in ["nam", "man", "male", "host", "teacher", "eric", "ryan", "mr", "david"]

        current_prompt = nam_prompt_feature if is_nam else nu_prompt_feature

        if text:
            # Báo cáo tiến độ ra màn hình web
            status_msg = f"🎙️ Đang thu âm câu [{index+1}/{total_lines}] của {clean_name.upper()}..."
            print(status_msg)
            yield status_msg, None

            try:
                with torch.inference_mode():
                    w, sr = model.generate_voice_clone(
                        text=text,
                        voice_clone_prompt=current_prompt
                    )

                temp_wav = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
                sf.write(temp_wav.name, w[0], sr)

                segment = AudioSegment.from_wav(temp_wav.name)
                final_audio += segment + gap
                os.unlink(temp_wav.name)

            except Exception as e:
                print(f"⚠️ Lỗi dòng {index+1}: {e}")

    yield "💾 Đang xuất file MP3 chất lượng cao...", None
    output_path = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3").name
    final_audio.export(output_path, format="mp3")

    yield "✅ HOÀN TẤT PODCAST!", output_path

# ================= GIAO DIỆN =================
with gr.Blocks(title="Speed Podcast Studio") as demo:
    gr.Markdown("# ⚡️ SPEED PODCAST STUDIO")
    gr.Markdown("Quy trình: **B1. Tạo giọng mẫu** -> **B2. Tải về máy** -> **B3. Upload lại vào B2 để render Podcast**.")

    with gr.Tab("B1: TẠO MẪU GIỌNG (Voice Design)"):
        with gr.Row():
            with gr.Column():
                gr.Markdown("### 👨 Tạo Mẫu Nam")
                nam_p = gr.Textbox(label="Mô tả giọng Nam", value="A professional male podcast host, deep and soothing voice. Laughing and energetic.")
                nam_s = gr.Textbox(label="Câu nói mẫu", value="Haha! Hello everyone, welcome to the show.")
                b_nam = gr.Button("Tạo & Nghe Thử")
                o_nam = gr.Audio(label="Kết quả Nam (Tải về nếu ưng ý)", type="filepath")

            with gr.Column():
                gr.Markdown("### 👩 Tạo Mẫu Nữ")
                nu_p = gr.Textbox(label="Mô tả giọng Nữ", value="A warm female voice, soft and velvety. American accent. Very happy.")
                nu_s = gr.Textbox(label="Câu nói mẫu", value="Wow! I am so excited to be here.")
                b_nu = gr.Button("Tạo & Nghe Thử")
                o_nu = gr.Audio(label="Kết quả Nữ (Tải về nếu ưng ý)", type="filepath")

    with gr.Tab("B2: RENDER PODCAST (Voice Clone)"):
        gr.Markdown("Upload file giọng mẫu (đã tạo ở B1 hoặc file có sẵn) vào đây để Clone.")
        with gr.Row():
            with gr.Column():
                r_nam = gr.Audio(label="Upload File Mẫu Nam", type="filepath")
                r_nu = gr.Audio(label="Upload File Mẫu Nữ", type="filepath")

            with gr.Column():
                script = gr.Textbox(lines=10, label="Kịch bản (Nam: ... / Nu: ...)",
                                  value="Nam: Haha! Welcome back.\nNu: Wow! This is fast.\nNam: Yes it is.")
                btn = gr.Button("🚀 RENDER SIÊU TỐC", variant="primary")
                status_text = gr.Markdown("⏳ Trạng thái: Đang chờ...") # Bảng báo cáo tiến độ thời gian thực
                out = gr.Audio(label="Kết quả Podcast")

    # Sự kiện
    b_nam.click(create_voice_sample, [nam_p, nam_s], [o_nam])
    b_nu.click(create_voice_sample, [nu_p, nu_s], [o_nu])

    # Nút Render kết nối với hàm Yield
    btn.click(generate_podcast_fast, [script, r_nam, r_nu], [status_text, out])

print("🌐 Đang khởi chạy...")
# Bật Queue siêu xịn của Gradio
demo.queue(api_open=False).launch(share=True, debug=True)

⏳ Đang cài đặt thư viện (Đợi 1-2 phút)...
✅ Cài đặt thư viện thành công!

********
********
 
🌐 Đang khởi chạy...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://53751da3b9f42d5a3b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
